# NB05a — Ablation study (all three datasets)

Retrains the COA configurations and their ablated variants (w/o COA, shallow ANN, w/o FastText, w/o BiLSTM, w/o dropout) with **one shared schedule** (ANN ≤50 epochs / patience 5, CNN ≤20 / patience 3; true labels for early stopping). Code identical to the tested ablation cell of the former NB05.

| | |
|---|---|
| **Required inputs** (Add Input → Your Work → Notebooks; always the newest version) | `01-preprocessing`, `02-coa-ann`, `03-coa-cnn`, `04-final-training` |
| **Optional inputs** | — |
| **Outputs** | `saved_results/{dataset}/{ann,cnn}_ablation.csv`, `ablation_reference_headline.json`, `saved_results/global/fig_ablation.pdf` |
| **Run time on Kaggle (estimate)** | ≈1.5 h typical, ≈4 h worst case (every model to its epoch limit) |
| **Accelerator / Internet** | GPU T4 x2 / off |

Together with NB05b this replaces the former `05-experiments` notebook (split so each part fits one session). Do not link `05-experiments` any more.

Step-by-step instructions for the whole pipeline: `Code/Newest/RUN_GUIDE.md`. Background: `Decision/012-split-notebooks-for-9h-sessions.md`.

In [ ]:
import os, random as _random, warnings, re, string, json, gc, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED       = 42
MAX_VOCAB  = 50000
EMBED_DIM  = 300
DATASETS   = ["amazon", "yelp", "twitter"]

WORKING     = "/kaggle/working"
SHARED_DIR  = f"{WORKING}/shared_data"
RESULTS_DIR = f"{WORKING}/saved_results"
MODELS_DIR  = f"{WORKING}/saved_models"

def ensure_dirs(dataset=None):
    for base in [SHARED_DIR, RESULTS_DIR, MODELS_DIR]:
        os.makedirs(base, exist_ok=True)
        if dataset:
            os.makedirs(f"{base}/{dataset}", exist_ok=True)
        else:
            for ds in DATASETS:
                os.makedirs(f"{base}/{ds}", exist_ok=True)

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    _random.seed(seed)
    np.random.seed(seed)

set_seeds()
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
    recall_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve, confusion_matrix)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import wilcoxon

tf.keras.mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()
N_GPUS = strategy.num_replicas_in_sync
print(f"MirroredStrategy: {N_GPUS} GPU(s)")

ANN_BATCH = 1024 * N_GPUS
CNN_BATCH = 128  * N_GPUS
print(f"ANN_BATCH={ANN_BATCH}  CNN_BATCH={CNN_BATCH}")

def set_seeds_tf(seed=SEED):
    set_seeds(seed)
    tf.random.set_seed(seed)

set_seeds_tf()
print(f"TF {tf.__version__}")

ensure_dirs()


In [ ]:
def evaluate(name, y_true, y_pred, y_prob=None):
    auc = round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    ap  = round(average_precision_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "Model"        : name,
        "Accuracy"     : round(accuracy_score(y_true, y_pred), 4),
        "Precision"    : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"       : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1-score"     : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "Macro-F1"     : round(rep["macro avg"]["f1-score"], 4),
        "Weighted-F1"  : round(rep["weighted avg"]["f1-score"], 4),
        "MCC"          : round(matthews_corrcoef(y_true, y_pred), 4),
        "AUC-ROC"      : auc,
        "Avg-Precision": ap,
    }

def best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2*prec*rec/(prec+rec+1e-9)
    idx = np.argmax(f1s[:-1])
    return float(thr[idx])

def make_es(patience=5, verbose=0):
    return EarlyStopping(monitor="val_loss", patience=patience,
                         restore_best_weights=True, verbose=verbose)

def make_tf_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.float32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def make_seq_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.int32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

print("Helpers ready.")


In [ ]:
def build_ann(h1, h2, lr, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(h2, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

def build_cnn(filters, lstm_units, lr, vocab_sz, emb_mat, max_len,
               seed=SEED, pretrained=True, use_bilstm=True, use_dropout=True):
    set_seeds_tf(seed)
    with strategy.scope():
        emb = (tf.keras.layers.Embedding(vocab_sz, EMBED_DIM,
                                          weights=[emb_mat], trainable=False)
               if pretrained else
               tf.keras.layers.Embedding(vocab_sz, EMBED_DIM, trainable=True))
        layers = [emb, tf.keras.layers.Conv1D(filters, 3, activation="relu"),
                  tf.keras.layers.MaxPooling1D(2)]
        if use_bilstm:
            layers.append(tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units)))
        else:
            layers.append(tf.keras.layers.GlobalMaxPooling1D())
        layers.append(tf.keras.layers.Dense(64, activation="relu"))
        if use_dropout:
            layers.append(tf.keras.layers.Dropout(0.5))
        layers.append(tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"))
        m = tf.keras.Sequential(layers)
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

def shallow_ann(h1, lr, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Model builders ready.")


In [ ]:
def load_dataset(dataset, nb01_shared):
    """Load all arrays for a given dataset."""
    d_dir = f"{nb01_shared}/{dataset}"
    cfg   = json.load(open(f"{d_dir}/config.json"))
    d = {
        "name"         : dataset,
        "cfg"          : cfg,
        "max_len"      : cfg["max_len"],
        "vocab_size"   : cfg["vocab_size"],
        "X_emb_train"  : np.load(f"{d_dir}/X_emb_train.npy"),
        "X_emb_val"    : np.load(f"{d_dir}/X_emb_val.npy"),
        "X_emb_test"   : np.load(f"{d_dir}/X_emb_test.npy"),
        "X_seq_train"  : np.load(f"{d_dir}/X_seq_train.npy"),
        "X_seq_val"    : np.load(f"{d_dir}/X_seq_val.npy"),
        "X_seq_test"   : np.load(f"{d_dir}/X_seq_test.npy"),
        "y_train"      : np.load(f"{d_dir}/y_train.npy"),
        "y_val"        : np.load(f"{d_dir}/y_val.npy"),
        "y_test"       : np.load(f"{d_dir}/y_test.npy"),
        "emb_matrix"   : np.load(f"{d_dir}/emb_matrix.npy"),
        "class_weights": {int(k):float(v) for k,v in
                          json.load(open(f"{d_dir}/class_weights.json")).items()},
    }
    with open(f"{d_dir}/test_texts_raw.txt", encoding="utf-8") as f:
        d["test_texts_raw"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_train.txt", encoding="utf-8") as f:
        d["X_txt_train"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_val.txt", encoding="utf-8") as f:
        d["X_txt_val"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_test.txt", encoding="utf-8") as f:
        d["X_txt_test"] = f.read().splitlines()
    print(f"  {dataset}: train={len(d['y_train']):,} val={len(d['y_val']):,} "
          f"test={len(d['y_test']):,} max_len={d['max_len']}")
    return d

def tv_combined(d):
    """Train+val combined for final training."""
    return {
        "X_emb": np.vstack([d["X_emb_train"], d["X_emb_val"]]),
        "X_seq": np.vstack([d["X_seq_train"], d["X_seq_val"]]),
        "y"    : np.concatenate([d["y_train"], d["y_val"]]),
        "X_txt": d["X_txt_train"] + d["X_txt_val"],
    }

print("Loader ready.")


## 1. Paths

In [ ]:
# ── UPDATE: replace username/slug with your actual Kaggle values ──────────────
NB01_DIR    = "/kaggle/input/notebooks/abhaysharma01702/01-preprocessing"
NB01_SHARED = f"{NB01_DIR}/shared_data"

print("Verifying NB01 shared_data...")
for ds in DATASETS:
    d = f"{NB01_SHARED}/{ds}"
    ok = os.path.isdir(d)
    n  = len(os.listdir(d)) if ok else 0
    cfg = json.load(open(f"{d}/config.json")) if ok else {}
    print(f"  {ds}: {'✓' if ok else '✗ MISSING'}  {n} files  "
          f"max_len={cfg.get('max_len','?')}  n_train={cfg.get('n_train','?')}")

NB02_DIR = "/kaggle/input/notebooks/abhaysharma01702/02-coa-ann"
NB03_DIR = "/kaggle/input/notebooks/abhaysharma01702/03-coa-cnn"
NB04_DIR = "/kaggle/input/notebooks/abhaysharma01702/04-final-training"
NB02_RESULTS = f"{NB02_DIR}/saved_results"
NB03_RESULTS = f"{NB03_DIR}/saved_results"
NB04_RESULTS = f"{NB04_DIR}/saved_results"

print("Verifying inputs...")
for ds in DATASETS:
    for rdir, name in [(NB02_RESULTS,"NB02"),(NB03_RESULTS,"NB03"),(NB04_RESULTS,"NB04")]:
        ok = os.path.isdir(f"{rdir}/{ds}")
        print(f"  {ds}/{name}: {'✓' if ok else '✗ MISSING'}")


In [ ]:
# ── Run control ───────────────────────────────────────────────────────────────
# This notebook is sized to finish well inside one 9-hour Kaggle session (see the header). As a safety net it saves
# every finished unit (one model/fold/seed) immediately and stops cleanly after TIME_LIMIT_HOURS. If that ever
# happens: save the version, add it as an input, list its slug in RESUME_SLUGS and run again — finished units are
# copied over and skipped.
import time, glob, shutil
RESUME_SLUGS     = []          # e.g. ["05a-ablation"]
TIME_LIMIT_HOURS = 8.0         # Kaggle sessions end at 9 h; the last hour is margin for loading and final cells
T_START = time.time()
def hours_used(): return (time.time() - T_START) / 3600

UNIT_DIR = f"{RESULTS_DIR}/units"
os.makedirs(UNIT_DIR, exist_ok=True)
for _slug in RESUME_SLUGS:
    _n = 0
    for _f in glob.glob(f"/kaggle/input/notebooks/abhaysharma01702/{_slug}/saved_results/units/*.json"):
        _dst = f"{UNIT_DIR}/{os.path.basename(_f)}"
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst); _n += 1
    print(f"Resumed {_n} finished units from {_slug}")

def unit_done(key): return os.path.exists(f"{UNIT_DIR}/{key}.json")
def unit_load(key):
    with open(f"{UNIT_DIR}/{key}.json") as fh:
        return json.load(fh)
def unit_save(key, obj):
    """Atomic write: a session killed mid-write cannot leave a truncated unit file behind."""
    tmp = f"{UNIT_DIR}/{key}.json.tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2)
    os.replace(tmp, f"{UNIT_DIR}/{key}.json")

class TimeLimit(Exception):
    pass
def check_time():
    if hours_used() > TIME_LIMIT_HOURS:
        raise TimeLimit()


## 2. Ablation study (all three datasets)

In [ ]:
# ── Ablation ──────────────────────────────────────────────────────────────────
# 2026-10-12: the "Full" rows are RETRAINED here with the same schedule as every ablated variant
# (ANN ≤50 epochs / patience 5, CNN ≤20 epochs / patience 3). Previously they were copied from NB04 (≤100/30 epochs),
# so Full-vs-variant differences mixed two effects. Early stopping here always used the true labels.
ABL_ANN = ["Full COA-ANN", "w/o COA (Default ANN)", "Shallow (1 layer)", "w/o FastText (zeros)"]
ABL_CNN = ["Full COA-CNN+BiLSTM", "w/o COA (Default CNN)", "w/o FastText (random)",
           "w/o BiLSTM (CNN only)", "w/o Dropout"]
ablation_all = {}

try:
    for ds in DATASETS:
        keys = ([f"abl_{ds}_ann_{i}" for i in range(len(ABL_ANN))] +
                [f"abl_{ds}_cnn_{i}" for i in range(len(ABL_CNN))])
        all_done = all(unit_done(k) for k in keys)
        ann_jobs = cnn_jobs = None
        if not all_done:
            print(f"\n{'='*60}\nABLATION | {ds.upper()}\n{'='*60}")
            d = load_dataset(ds, NB01_SHARED)
            ml = d["max_len"]; vs = d["vocab_size"]; emb = d["emb_matrix"]
            ann_cfg = json.load(open(f"{NB02_RESULTS}/{ds}/coa_ann.json"))
            cnn_cfg = json.load(open(f"{NB03_RESULTS}/{ds}/coa_cnn.json"))
            h1=ann_cfg["best_h1"]; h2=ann_cfg["best_h2"]; ann_lr=ann_cfg["best_lr"]
            f_=cnn_cfg["best_filters"]; lstm_u=cnn_cfg["best_lstm_units"]; cnn_lr=cnn_cfg["best_lr"]
            tv = tv_combined(d)
            cw_tv = dict(zip([int(k) for k in np.unique(tv["y"])],
                             [float(v) for v in compute_class_weight("balanced",
                                                 classes=np.unique(tv["y"]), y=tv["y"])]))
            n_es = int(len(tv["y"])*0.1)

            def fit_ann(m, zero_input=False):
                Xtv = np.zeros_like(tv["X_emb"]) if zero_input else tv["X_emb"]
                Xte = np.zeros_like(d["X_emb_test"]) if zero_input else d["X_emb_test"]
                m.fit(make_tf_ds(Xtv[:-n_es], tv["y"][:-n_es], ANN_BATCH),
                      validation_data=make_tf_ds(Xtv[-n_es:], tv["y"][-n_es:], ANN_BATCH, shuffle=False),
                      epochs=50, callbacks=[make_es(5)], class_weight=cw_tv, verbose=0)
                return m.predict(make_tf_ds(Xte, np.zeros(len(d["y_test"])), ANN_BATCH, shuffle=False),
                                 verbose=0).flatten()

            def fit_cnn(m):
                m.fit(make_seq_ds(tv["X_seq"][:-n_es], tv["y"][:-n_es], CNN_BATCH),
                      validation_data=make_seq_ds(tv["X_seq"][-n_es:], tv["y"][-n_es:], CNN_BATCH, shuffle=False),
                      epochs=20, callbacks=[make_es(3)], class_weight=cw_tv, verbose=0)
                return m.predict(make_seq_ds(d["X_seq_test"], np.zeros(len(d["y_test"])), CNN_BATCH, shuffle=False),
                                 verbose=0).flatten()

            ann_jobs = [lambda: fit_ann(build_ann(h1, h2, ann_lr)),
                        lambda: fit_ann(build_ann(128, 64, 1e-3)),
                        lambda: fit_ann(shallow_ann(h1, ann_lr)),
                        lambda: fit_ann(build_ann(h1, h2, ann_lr), zero_input=True)]
            cnn_jobs = [lambda: fit_cnn(build_cnn(f_, lstm_u, cnn_lr, vs, emb, ml)),
                        lambda: fit_cnn(build_cnn(128, 64, 1e-3, vs, emb, ml)),
                        lambda: fit_cnn(build_cnn(f_, lstm_u, cnn_lr, vs, emb, ml, pretrained=False)),
                        lambda: fit_cnn(build_cnn(f_, lstm_u, cnn_lr, vs, emb, ml, use_bilstm=False)),
                        lambda: fit_cnn(build_cnn(f_, lstm_u, cnn_lr, vs, emb, ml, use_dropout=False))]

        results = {"ann": [], "cnn": []}
        for arch, names, jobs in [("ann", ABL_ANN, ann_jobs), ("cnn", ABL_CNN, cnn_jobs)]:
            for i, name in enumerate(names):
                key = f"abl_{ds}_{arch}_{i}"
                if not unit_done(key):
                    check_time()
                    print(f"  [{arch.upper()} {i+1}/{len(names)}] {name}...")
                    p = jobs[i]()
                    unit_save(key, evaluate(name, d["y_test"], (p > 0.5).astype(int), p))
                    tf.keras.backend.clear_session(); gc.collect()
                r = unit_load(key); results[arch].append(r)
                print(f"    {name}: F1={r['F1-score']}  MCC={r['MCC']}")

        ablation_all[ds] = results
        r_dir = f"{RESULTS_DIR}/{ds}"; os.makedirs(r_dir, exist_ok=True)
        pd.DataFrame(results["ann"]).to_csv(f"{r_dir}/ann_ablation.csv", index=False)
        pd.DataFrame(results["cnn"]).to_csv(f"{r_dir}/cnn_ablation.csv", index=False)
        json.dump({"note": "NB04 headline runs (<=100/30 epochs); NOT comparable with the ablation rows.",
                   "COA-ANN": json.load(open(f"{NB04_RESULTS}/{ds}/ann_coa_metrics.json")),
                   "COA-CNN+BiLSTM": json.load(open(f"{NB04_RESULTS}/{ds}/cnn_coa_metrics.json"))},
                  open(f"{r_dir}/ablation_reference_headline.json", "w"), indent=2)
        if not all_done:
            del d, tv, emb; gc.collect()
        print(f"  Saved ablation to {r_dir}/")
except TimeLimit:
    print(f"\nTime limit ({TIME_LIMIT_HOURS} h) reached during the ablation. Finished units are saved; "
          "link this version, set RESUME_SLUGS and run again.")

# Plot ablation (datasets that are complete)
done = [ds for ds in DATASETS if ds in ablation_all]
if done:
    fig, axes = plt.subplots(2, len(done), figsize=(6 * len(done), 10), squeeze=False)
    for col, ds in enumerate(done):
        for row, (key, title) in enumerate([("ann", "ANN"), ("cnn", "CNN+BiLSTM")]):
            df = pd.DataFrame(ablation_all[ds][key])
            ax = axes[row, col]
            x = np.arange(len(df)); w = 0.2
            for i, (m_, c_) in enumerate(zip(["F1-score", "Macro-F1", "MCC", "AUC-ROC"],
                                             ["#e74c3c", "#2ecc71", "#f39c12", "#3498db"])):
                ax.bar(x + i * w, df[m_], w, label=m_, color=c_, alpha=0.85)
            ax.set_xticks(x + 1.5 * w)
            ax.set_xticklabels(df["Model"], rotation=15, ha="right", fontsize=7)
            ax.set_ylim(0.0, 1.05); ax.set_title(f"{title} Ablation — {ds.capitalize()}", fontsize=9)
            ax.set_ylabel("Score"); ax.legend(fontsize=7); ax.grid(axis="y", alpha=0.3)
    plt.suptitle("Ablation Study — all rows trained on the same schedule", fontsize=13)
    plt.tight_layout()
    os.makedirs(f"{RESULTS_DIR}/global", exist_ok=True)
    plt.savefig(f"{RESULTS_DIR}/global/fig_ablation.pdf", dpi=300, bbox_inches="tight")
    plt.show()
